# Rate laws and the Arrhenius plot

*physchem · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/arrhenius_kinetics)

How fast does a reaction go, and how much faster when it is warmer? The zeroth-, first- and second-order rate laws are integrated for six temperatures at once in one $(O, T)$ tensor, with $k(T) = A\,e^{-E_a/RT}$. A noisy synthetic experiment then reveals its order (which plot of $[A]$, $\ln[A]$ or $1/[A]$ is straight?) and its activation energy (a least-squares line through $\ln k$ against $1/T$).

**Vector form**

$$-\frac{d[\mathrm A]}{dt} = k(T)\,[\mathrm A]^{n},\qquad k(T) = A\,e^{-E_a/RT},\qquad \ln k = \ln A - \frac{E_a}{R}\,\frac{1}{T}$$

**Index (tensor) form**

$$\frac{d[\mathrm A]_{ot}}{dt} = -k_t\,[\mathrm A]_{ot}^{\,n_o} \qquad k_t = A\,e^{-E_a/(R T_t)} \qquad G_{ij} = X_{ti}X_{tj},\;\; b_i = X_{ti}\,y_t,\;\; G_{ij}\beta_j = b_i$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_t$ | `T` | (T,) | six temperatures between $T_{min}$ and $T_{max}$ (K) |
| $n_o$ | `n` | (O,) | reaction orders $0, 1, 2$, integrated side by side |
| $k_t$ | `rate_constant(T, A, Ea)` | (T,) | Arrhenius rate constant, in $\mathrm{M}^{1-n}\,\mathrm{s^{-1}}$ |
| $k_t [\mathrm A]_{ot}^{n_o}$ | `einsum('t,ot->ot', k, conc ** n[:, None])` | (O, T) | rate of every run; no index summed, $k_t$ is broadcast over the orders |
| $t_{\mathrm{end},ot}$ | `run_lengths(k, n)` | (O, T) | each run is followed to 90 % conversion, on its own clock $t = \tau\,t_{end}$ |
| $[\mathrm A]_{ot}(\tau)$ | `integrate(rhs, conc0, 1/N, N)` | (N+1, O, T) | RK4 trajectory, compared with the exact integrated laws in the tests |
| $g = [\mathrm A],\ \ln[\mathrm A],\ 1/[\mathrm A]$ | `linearize(data)` | (3, T, S) | the three plots against $t$; only the right order gives a straight line |
| $X_{si} = (1, t_s)$ | `torch.stack([ones, x], -1)` | (…, S, 2) | design matrix of a straight-line fit |
| $G_{ij} = w_s X_{si} X_{sj}$ | `einsum('...s,...si,...sj->...ij', w, X, X)` | (…, 2, 2) | normal-equation matrix; $s$ (the samples) summed; weights $w_s = (dg/d[A])^{-2}$, and $w_s = 0$ drops points below 2 % of $[A]_0$ |
| $\beta_j$ | `torch.linalg.solve(G, b)` | (…, 2) | intercept and slope, for every plot and temperature at once |
| $y_t = \ln k_t,\ x_t = 1/T_t$ | `arrhenius_fit(T, k_fit)` | (2,) | Arrhenius line: slope $-E_a/R$, intercept $\ln A$ |

Each rate law has an integrated form that turns into a straight line in the right variable: $[A] = [A]_0 - kt$ (zeroth order), $\ln[A] = \ln[A]_0 - kt$ (first) and $1/[A] = 1/[A]_0 + kt$ (second). So a chemist plots all three and keeps the straightest one ($R^2$ closest to 1); its slope is $\mp k$. A second test uses half-lives: since $t_{1/2} \propto [A]_0^{1-n}$, the time from $[A]_0/2$ to $[A]_0/4$ is $2^{n-1}$ times the first half-life (shorter, equal or twice as long), which gives $n = 1 + \log_2(t_{1/2}^{(2)}/t_{1/2}^{(1)})$. Repeating the run at several temperatures gives $k(T)$, and the Arrhenius plot of $\ln k$ against $1/T$ is a straight line of slope $-E_a/R$. With $E_a = 75$ kJ/mol, warming from 290 K to 300 K roughly speeds the reaction up 2.8 times; the old rule of thumb "twice as fast per 10 K" corresponds to $E_a \approx 50$ kJ/mol near room temperature. Each run is followed until 90 % of A has reacted, so cold runs last much longer than hot ones (see the legend); the left plot uses the fraction of each run's time so that all six fit on one axis, which does not change which plot is straight. The data are synthetic (a made-up reaction, Gaussian noise with a fixed seed). Taking $\ln$ or $1/x$ magnifies the noise of late, dilute points ($\sigma/[A]$ and $\sigma/[A]^2$), so each point is weighted by $(dg/d[A])^{-2}$, i.e. by $1$, $[A]^2$ or $[A]^4$, and points below 2 % of $[A]_0$ are left out. Try a larger noise to see the $R^2$ test become less decisive, or a narrower temperature range to see the error on $E_a$ grow.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/arrhenius_kinetics/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

NAMES = ("[A]", "ln[A]", "1/[A]")
SIGN = torch.tensor([-1.0, -1.0, 1.0], dtype=tensors.DTYPE)  # slope = -k, -k, +k for orders 0, 1, 2


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    Ea = params["Ea"] * 1e3  # kJ/mol → J/mol
    A = 10 ** params["log10_A"]
    true_order = int(params["order"])

    T = tensors.temperatures(params["T_min"], params["T_max"], dev)  # (T,) K
    k = equation.rate_constant(T, A, Ea)  # (T,)
    n = tensors.orders(dev)  # (O,)
    t_end = tensors.run_lengths(k, n)  # (O, T) s
    conc0 = tensors.initial_state(len(n), len(T), dev)  # (O, T)

    # 1) integrate all three rate laws at all temperatures at once
    tau, traj = solve.integrate_runs(conc0, k, n, t_end)  # (N+1,), (N+1, O, T)

    # 2) half-life method on the smooth curves: t½ from [A]_0 → [A]_0/2, then from [A]_0/2 → [A]_0/4
    t_50 = solve.crossing_time(tau, traj, t_end, 0.5 * tensors.A0)
    t_25 = solve.crossing_time(tau, traj, t_end, 0.25 * tensors.A0)
    n_half = equation.order_from_half_lives(t_50, t_25 - t_50)  # (O, T)
    print("half-life method on the integrated curves (should give 0, 1, 2):")
    for o in range(len(n)):
        ratio = ((t_25 - t_50) / t_50)[o, 0].item()
        print(f"  rate law n = {o}:  t½(2)/t½(1) = {ratio:.3f}  →  n = {n_half[o].mean().item():.3f}")

    # 3) a noisy "experiment" on the reaction of unknown order: which plot is straight?
    tau_s, clean = solve.sample(tau, traj, true_order)  # (S,), (T, S)
    data = tensors.measure(clean, params["noise"])  # (T, S)
    times = tau_s[None, :] * t_end[true_order][:, None]  # (T, S) real sampling times, s
    beta, R2 = solve.fit_linearizations(times, data)  # (3, T, 2), (3, T)
    best = int(R2.mean(1).argmax())
    print(f"\nmeasured run, noise σ = {params['noise']:.3f} M — mean R² of each straight-line plot:")
    for g in range(3):
        mark = "  ← straightest" if g == best else ""
        print(f"  {NAMES[g]:<6} vs t:  R² = {R2[g].mean().item():.5f}{mark}")
    print(f"  ⇒ reaction order n = {best} (true: {true_order})")

    # 4) k from the slope of the straight plot, then the Arrhenius fit
    k_fit = SIGN[best].item() * beta[best, :, 1]  # (T,)
    Ea_fit, A_fit = equation.arrhenius_fit(T, k_fit.clamp_min(1e-300))
    print("\n   T (K)    k true       k fit        run length")
    for t in range(len(T)):
        run_time = plot.fmt_time(t_end[true_order, t].item())
        print(f"  {T[t].item():6.1f}  {k[t].item():.4e}  {k_fit[t].item():.4e}  {run_time}")
    print(f"\nArrhenius fit:  E_a = {Ea_fit / 1e3:.2f} kJ/mol (true {Ea / 1e3:.2f})   A = {A_fit:.3e} (true {A:.3e})")

    return plot.animate(
        tau, traj[:, true_order], t_end[true_order], T, tau_s, data, beta, R2, best, k, k_fit, Ea_fit, A_fit,
        out_dir / "arrhenius_kinetics.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/arrhenius_kinetics/equation.py
r"""Physics → tensor translation.

A reaction A → products with rate law of order n and a temperature-dependent rate constant (Arrhenius):

    r_{ot} = k_t [A]_{ot}^{n_o},        k_t = A e^{-E_a / (R T_t)},        d[A]_{ot}/dt = -r_{ot}

o runs over the three orders n = 0, 1, 2 and t over the temperatures; no index is summed in the rate law,
every (o, t) pair is its own experiment. Each run has its own clock, t = τ t_end,ot with τ ∈ [0, 1], so the
integrator steps in τ:

    d[A]_{ot}/dτ = -t_end,ot k_t [A]_{ot}^{n_o}

Fitting a straight line y_s ≈ β_0 + β_1 x_s by least squares is also tensor algebra (normal equations):

    G_{ij} = w_s X_{si} X_{sj},    b_i = w_s X_{si} y_s,    G_{ij} β_j = b_i        (s and j summed)

with design matrix X_{si} = (1, x_s) and weights w_s (1 for a point that counts, 0 for one left out).
"""

import math

import torch

R_GAS = 8.314462618  # J/(mol K)


def rate_constant(T: torch.Tensor, A: float, Ea: float) -> torch.Tensor:
    """Arrhenius law k_t = A exp(-E_a / (R T_t)).   T: (T,) → (T,).

    Units of k: M^(1-n) s⁻¹; the same number is used for every order n.
    """
    return A * torch.exp(-Ea / (R_GAS * T))


def rate(conc: torch.Tensor, k: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    """r_{ot} = k_t [A]_{ot}^{n_o}.   conc: (O, T), k: (T,), n: (O,) → (O, T).  Nothing is summed."""
    return torch.einsum("t,ot->ot", k, conc ** n[:, None])


def rhs(conc: torch.Tensor, k: torch.Tensor, n: torch.Tensor, t_end: torch.Tensor) -> torch.Tensor:
    """d[A]/dτ = -t_end r for the state (O, T) in reduced time τ = t / t_end."""
    return -t_end * rate(conc, k, n)


def integrated_law(t: torch.Tensor, A0: float, k: torch.Tensor, n: int) -> torch.Tensor:
    """The textbook integrated rate laws (exact solutions), for comparison and for the half-life method.

    n = 0:  [A] = [A]_0 - k t               (straight line in [A])
    n = 1:  [A] = [A]_0 e^{-k t}             (straight line in ln[A])
    n = 2:  [A] = [A]_0 / (1 + [A]_0 k t)    (straight line in 1/[A])
    """
    if n == 0:
        return A0 - k * t
    if n == 1:
        return A0 * torch.exp(-k * t)
    return A0 / (1.0 + A0 * k * t)


def time_to_fraction(k: torch.Tensor, n: torch.Tensor, A0: float, f: float) -> torch.Tensor:
    """Time for [A] to fall from [A]_0 to f [A]_0, from the integrated laws (n may be a tensor of orders).

    n = 0: (1 - f) [A]_0 / k     n = 1: ln(1/f) / k     n = 2: (1/f - 1) / ([A]_0 k)
    """
    zero = (1.0 - f) * A0 / k
    first = math.log(1.0 / f) / k
    second = (1.0 / f - 1.0) / (A0 * k)
    return torch.where(n == 0, zero, torch.where(n == 1, first, second))


def linearize(conc: torch.Tensor) -> torch.Tensor:
    """Stack the three candidate straight-line plots: g = ([A], ln[A], 1/[A]).   (...) → (3, ...)."""
    safe = conc.clamp_min(1e-6)  # a noisy point can dip below zero near the end
    return torch.stack([conc, torch.log(safe), 1.0 / safe])


def least_squares(x: torch.Tensor, y: torch.Tensor, w: torch.Tensor | None = None) -> torch.Tensor:
    r"""Fit y ≈ β_0 + β_1 x along the last axis, batched over any leading axes.   Returns β (..., 2).

    X_{si} = (1, x_s);  G_{ij} = w_s X_{si} X_{sj};  b_i = w_s X_{si} y_s;  solve G β = b   (s summed).
    """
    if w is None:
        w = torch.ones_like(y)
    X = torch.stack([torch.ones_like(x), x], dim=-1)  # (..., S, 2)
    G = torch.einsum("...s,...si,...sj->...ij", w, X, X)  # (..., 2, 2)
    b = torch.einsum("...s,...si,...s->...i", w, X, y)  # (..., 2)
    return torch.linalg.solve(G, b)


def r_squared(x: torch.Tensor, y: torch.Tensor, beta: torch.Tensor, w: torch.Tensor | None = None) -> torch.Tensor:
    """Coefficient of determination R² = 1 - SS_res / SS_tot of the fitted line (1 = perfectly straight)."""
    if w is None:
        w = torch.ones_like(y)
    y_fit = beta[..., :1] + beta[..., 1:] * x
    mean = (w * y).sum(-1, keepdim=True) / w.sum(-1, keepdim=True)
    ss_res = (w * (y - y_fit) ** 2).sum(-1)
    ss_tot = (w * (y - mean) ** 2).sum(-1)
    return 1.0 - ss_res / ss_tot


def arrhenius_fit(T: torch.Tensor, k: torch.Tensor) -> tuple[float, float]:
    """Straight line through the Arrhenius plot: ln k_t = ln A - (E_a/R)(1/T_t).   Returns (E_a in J/mol, A)."""
    beta = least_squares(1.0 / T, torch.log(k))  # β_0 = ln A, β_1 = -E_a / R
    return -R_GAS * beta[1].item(), math.exp(beta[0].item())


def order_from_half_lives(t_half_1: torch.Tensor, t_half_2: torch.Tensor) -> torch.Tensor:
    """n = 1 + log₂(t½⁽²⁾ / t½⁽¹⁾): since t½ ∝ [A]_0^{1-n}, the second half-life (starting from [A]_0/2)
    is 2^{n-1} times the first: shorter for n = 0, equal for n = 1, twice as long for n = 2."""
    return 1.0 + torch.log2(t_half_2 / t_half_1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/arrhenius_kinetics/tensors.py
"""Build the concrete tensors: temperatures, rate constants, reaction orders and run lengths.

Units: concentrations in mol/L (M), times in s, energies in J/mol. The data are synthetic: a made-up reaction
with a textbook-scale activation energy (50–100 kJ/mol), "measured" with a fixed random seed.
"""

import torch

from . import equation

ORDERS = (0, 1, 2)  # the three rate laws integrated side by side
A0 = 1.0  # initial concentration [A]_0 (M), the same in every run
N_TEMPS = 6  # number of temperatures (one batch entry each)
N_SAMPLES = 20  # "measurements" per run
DTYPE = torch.float64  # least squares on ln k likes double precision


def temperatures(T_min: float, T_max: float, device: torch.device) -> torch.Tensor:
    """T_t in K, shape (T,): evenly spaced between T_min and T_max (at least 5 K apart)."""
    T_max = max(T_max, T_min + 5.0)
    return torch.linspace(T_min, T_max, N_TEMPS, device=device, dtype=DTYPE)


def orders(device: torch.device) -> torch.Tensor:
    """n_o, shape (O,) = (3,): zeroth, first and second order."""
    return torch.tensor(ORDERS, device=device, dtype=DTYPE)


def run_lengths(k: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    """t_end,ot, shape (O, T): every run is followed until 90 % of A is used up ([A] = 0.1 [A]_0).

    A cold run is much slower than a hot one, so each (order, temperature) pair gets its own clock,
    just as a chemist would sample a slow run less often.
    """
    return equation.time_to_fraction(k[None, :], n[:, None], A0, 0.1)


def initial_state(n_orders: int, n_temps: int, device: torch.device) -> torch.Tensor:
    """[A]_{ot} at τ = 0, shape (O, T): every run starts at [A]_0."""
    return torch.full((n_orders, n_temps), A0, device=device, dtype=DTYPE)


def measure(conc: torch.Tensor, noise: float, seed: int = 0) -> torch.Tensor:
    """Add Gaussian measurement noise (σ = noise, in M) with a fixed seed.   conc: (T, S) → (T, S)."""
    gen = torch.Generator().manual_seed(seed)  # CPU generator: the same numbers on every device
    eps = torch.randn(conc.shape, generator=gen, dtype=DTYPE).to(conc.device)
    return conc + noise * eps


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/arrhenius_kinetics/solve.py
"""Integrate all rate laws with RK4, then analyse the "measured" run: which order, which k, which E_a."""

import torch

from problems._common.integrators import integrate

from . import equation
from .tensors import N_SAMPLES

N_STEPS = 400  # RK4 steps in reduced time τ ∈ [0, 1]


def integrate_runs(
    conc0: torch.Tensor, k: torch.Tensor, n: torch.Tensor, t_end: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (τ (N+1,), [A] (N+1, O, T)). All orders and temperatures advance together in one (O, T) tensor."""
    traj = integrate(lambda c, tau: equation.rhs(c, k, n, t_end), conc0, 1.0 / N_STEPS, N_STEPS)
    tau = torch.linspace(0.0, 1.0, N_STEPS + 1, device=conc0.device, dtype=conc0.dtype)
    return tau, traj


def crossing_time(tau: torch.Tensor, traj: torch.Tensor, t_end: torch.Tensor, level: float) -> torch.Tensor:
    """Time at which the (decreasing) [A] first drops to `level`, by linear interpolation.   → (O, T)."""
    i = ((traj > level).sum(0) - 1).clamp(0, len(tau) - 2)  # last grid point still above the level
    c_lo = traj.gather(0, i[None]).squeeze(0)
    c_hi = traj.gather(0, (i + 1)[None]).squeeze(0)
    frac = (c_lo - level) / (c_lo - c_hi)
    return (tau[i] + frac * (tau[1] - tau[0])) * t_end


def sample(tau: torch.Tensor, traj: torch.Tensor, order: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Pick N_SAMPLES evenly spaced points of the run with order `order`.   Returns (τ_s (S,), [A]_{ts} (T, S))."""
    idx = torch.linspace(0, len(tau) - 1, N_SAMPLES, device=tau.device).round().long()
    return tau[idx], traj[idx, order].T


def fit_linearizations(times: torch.Tensor, data: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Straight-line fits of [A], ln[A] and 1/[A] against t for every temperature.

    times, data: (T, S). Returns β (3, T, 2) and R² (3, T). The noise σ on [A] becomes σ·|dg/d[A]| on the plotted
    quantity g (σ/[A] for ln[A], σ/[A]² for 1/[A]), so each point is weighted by w = (dg/d[A])⁻² = (1, [A]², [A]⁴):
    late, low-concentration points count less. Points below 2 % of [A]_0 are left out entirely (w = 0).
    """
    g = equation.linearize(data)  # (3, T, S)
    keep = (data > 0.02).to(data.dtype)  # (T, S)
    c = data.clamp_min(0.02)
    w = keep * torch.stack([torch.ones_like(c), c**2, c**4])  # (3, T, S) error-propagation weights
    x = times.expand_as(g)
    beta = equation.least_squares(x, g, w)
    return beta, equation.r_squared(x, g, beta, w)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/arrhenius_kinetics/plot.py
"""Animate the runs: [A](t) at every temperature, the three straight-line tests, and the Arrhenius plot."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import MaxNLocator

from problems._common.plotting import frame_indices, save_gif

from .equation import R_GAS

N_FRAMES = 50  # frames on a log-spaced common clock
LABELS = ("[A] (zeroth order)", "ln[A] (first order)", "1/[A] (second order)")


def fmt_time(seconds: float) -> str:
    if seconds < 120:
        return f"{seconds:.3g} s"
    if seconds < 7200:
        return f"{seconds / 60:.3g} min"
    if seconds < 3 * 86400:
        return f"{seconds / 3600:.3g} h"
    return f"{seconds / 86400:.3g} d"


def animate(
    tau: torch.Tensor,
    conc: torch.Tensor,
    t_end: torch.Tensor,
    T: torch.Tensor,
    tau_s: torch.Tensor,
    data: torch.Tensor,
    beta: torch.Tensor,
    R2: torch.Tensor,
    best: int,
    k_true: torch.Tensor,
    k_fit: torch.Tensor,
    Ea_fit: float,
    A_fit: float,
    path: Path,
) -> Path:
    """conc (N+1, T): smooth curves of the measured order; data (T, S): noisy samples at τ_s (S,)."""
    tau, conc, t_end, T = (x.cpu().numpy() for x in (tau, conc, t_end, T))
    tau_s, data, beta, R2 = (x.cpu().numpy() for x in (tau_s, data, beta, R2))
    k_true, k_fit = k_true.cpu().numpy(), k_fit.cpu().numpy()
    n_T = len(T)
    colors = plt.get_cmap("plasma")(np.linspace(0.0, 0.85, n_T))  # dark = cold, orange = hot
    g_data = np.stack([data, np.log(np.clip(data, 1e-6, None)), 1 / np.clip(data, 1e-6, None)])  # (3, T, S)

    fig = plt.figure(figsize=(9.0, 4.4))
    grid = fig.add_gridspec(3, 3, width_ratios=[1.15, 0.9, 1.0])
    ax_c = fig.add_subplot(grid[:, 0])
    ax_g = [fig.add_subplot(grid[i, 1]) for i in range(3)]
    ax_a = fig.add_subplot(grid[:, 2])

    # --- concentration curves against real time (log axis): hot runs finish first ---
    t_real = tau[:, None] * t_end[None, :]  # (N+1, T) s
    t_samp = tau_s[None, :] * t_end[:, None]  # (T, S) s
    clock = np.geomspace(0.02 * t_end.min(), t_end.max(), N_FRAMES)  # one common clock for the animation
    ax_c.set_xscale("log")
    ax_c.minorticks_off()  # fewer tick artists to redraw in every frame
    ax_c.set_xlim(clock[0], clock[-1] * 1.05)
    ax_c.set_ylim(0, 1.08)
    ax_c.set_xlabel("time $t$ (s, log scale)")
    ax_c.set_ylabel("[A] (M)")
    ax_c.set_title("measured runs: warmer = faster", fontsize=10)
    curves = [ax_c.plot([], [], lw=1.6, color=colors[t], label=f"{T[t]:.0f} K")[0] for t in range(n_T)]
    dots_c = [ax_c.plot([], [], "o", ms=3, color=colors[t])[0] for t in range(n_T)]
    ax_c.legend(fontsize=7, loc="lower left")
    time_text = ax_c.text(0.97, 0.97, "", transform=ax_c.transAxes, ha="right", va="top", fontsize=9)

    # --- the three candidate straight lines ---
    dots_g = []
    for g, ax in enumerate(ax_g):
        ok = "  ✓" if g == best else ""
        ax.set_title(f"{LABELS[g]}  R² = {R2[g].mean():.4f}{ok}", fontsize=8, pad=2,
                     color="#15803d" if g == best else "#374151")  # fmt: skip
        ax.set_xlim(0, 1)
        ax.tick_params(labelsize=7)
        ax.yaxis.set_major_locator(MaxNLocator(3))  # few ticks: fewer artists to redraw per frame
        lo, hi = np.nanmin(g_data[g]), np.nanmax(g_data[g])
        ax.set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
        for t in range(n_T):  # fitted line, drawn back in reduced time: y = β0 + β1 t_end τ
            ax.plot(tau, beta[g, t, 0] + beta[g, t, 1] * t_end[t] * tau, lw=0.8, color=colors[t], alpha=0.6)
        dots_g.append([ax.plot([], [], "o", ms=2.2, color=colors[t])[0] for t in range(n_T)])
        if g < 2:
            ax.set_xticklabels([])
    ax_g[2].set_xlabel(r"$t\,/\,t_{\mathrm{end}}$  (each run to 90 % conversion)", fontsize=8)

    # --- Arrhenius plot: ln k against 1000/T ---
    x = 1000.0 / T
    x_line = np.linspace(x.min() - 0.03, x.max() + 0.03, 50)
    y_fit = np.log(np.clip(k_fit, 1e-300, None))
    (line,) = ax_a.plot([], [], color="#374151", lw=1.4, label="LSQ fit")
    ax_a.plot(x, np.log(k_true), "x", color="#9ca3af", ms=6, label="true $k$")
    points = ax_a.scatter(x, y_fit, c=colors, s=28, zorder=3, label="fitted $k$")
    ax_a.set_xlim(x_line[0], x_line[-1])
    ax_a.set_ylim(min(y_fit.min(), np.log(k_true).min()) - 0.4, max(y_fit.max(), np.log(k_true).max()) + 0.4)
    ax_a.set_xlabel("1000 / T  (1/K)")
    ax_a.set_ylabel(r"$\ln k$")
    ax_a.set_title("Arrhenius plot", fontsize=10)
    fit_text = ax_a.text(0.97, 0.97, f"slope = $-E_a/R$\n$E_a$ = {Ea_fit / 1e3:.1f} kJ/mol\n$A$ = {A_fit:.2e}",
              transform=ax_a.transAxes, ha="right", va="top", fontsize=8)  # fmt: skip
    ax_a.legend(fontsize=7, loc="lower left")

    def update(step: int):
        now = clock[step]
        for t in range(n_T):
            i = int(np.searchsorted(t_real[:, t], now, side="right"))  # grid points of run t done by now
            n_shown = int(np.searchsorted(t_samp[t], now, side="right"))  # samples measured so far
            curves[t].set_data(t_real[1:i, t], conc[1:i, t])  # skip t = 0, which a log axis cannot show
            dots_c[t].set_data(t_samp[t, 1:n_shown], data[t, 1:n_shown])
            for g in range(3):
                dots_g[g][t].set_data(tau_s[:n_shown], g_data[g, t, :n_shown])
        done = now >= t_end * (1 - 1e-9)  # runs that have finished: their k goes on the Arrhenius plot
        points.set_offsets(np.c_[x, np.where(done, y_fit, np.nan)])
        all_done = bool(done.all())  # the straight-line fit needs every temperature
        line.set_data(x_line, np.log(A_fit) - Ea_fit / R_GAS * x_line / 1000.0)
        line.set_visible(all_done)
        fit_text.set_visible(all_done)
        time_text.set_text(f"t = {fmt_time(now)}")

    fig.tight_layout(pad=0.6, w_pad=0.8, h_pad=0.4)
    return save_gif(fig, update, frame_indices(N_FRAMES), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'Ea': 75.0,
    'log10_A': 11.0,
    'order': '1',
    'T_min': 290.0,
    'T_max': 340.0,
    'noise': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.arrhenius_kinetics")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.arrhenius_kinetics.main")
problem_dir = Path("problems/physchem/arrhenius_kinetics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
